# Setup

In [1]:
from pprint import pprint
from collections import deque
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def merge_sort(lista, index=0, ascending=True):
    """
    Implementação do Merge Sort para listas de tuplas ou listas.
    :param lista: Lista de itens a serem ordenados.
    :param index: Índice do elemento dentro da tupla/lista usado como chave.
    :param ascending: True para ordem crescente, False para decrescente.
    """
    if len(lista) <= 1:
        return lista

    meio = len(lista) // 2
    esquerda = merge_sort(lista[:meio], index, ascending)
    direita = merge_sort(lista[meio:], index, ascending)

    return merge(esquerda, direita, index, ascending)

def merge(esquerda, direita, index, ascending):
    resultado = []
    i = j = 0

    while i < len(esquerda) and j < len(direita):
        # Lógica de comparação baseada no índice e direção
        if ascending:
            condicao = esquerda[i][index] <= direita[j][index]
        else:
            condicao = esquerda[i][index] >= direita[j][index]

        if condicao:
            resultado.append(esquerda[i])
            i += 1
        else:
            resultado.append(direita[j])
            j += 1

    resultado.extend(esquerda[i:])
    resultado.extend(direita[j:])
    return resultado

# Par

In [2]:
import pandas as pd
from collections import deque
from pprint import pprint
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# OBS: Certifique-se de ter a função 'merge_sort' definida em uma célula anterior!

# ------------------------------------------------------------
# 1. CARREGAMENTO DOS DADOS
# Lê o CSV e converte cada linha em uma TUPLA imutável
# representando um pedido: (id, cidade, produto, categoria,
# quantidade, valor_unitario, urgencia, tempo, modal, pagamento)
# Big O: O(n) — percorre todas as n linhas do CSV uma vez
# ------------------------------------------------------------
df_par = pd.read_csv("Check_point_1_dados_logistica_RA_final_par.csv")
print("=== DataFrame Original ===")
display(df_par)

# Converte cada linha do DataFrame em uma tupla imutável
# Tuplas são usadas aqui pois os dados do pedido não devem ser alterados
pedidos_tuplas = [tuple(row) for row in df_par.itertuples(index=False, name=None)]

# Mapeamento das colunas para facilitar acesso por índice
# (pedido_id, cidade_destino, produto, categoria, quantidade,
#  valor_unitario, urgencia, tempo_estimado_horas, modal, status_pagamento)
print("\n=== Pedidos como Tuplas ===")
pprint(pedidos_tuplas)


# ------------------------------------------------------------
# 2. ORDENAÇÃO DOS PEDIDOS
# Ordena os pedidos pelo valor unitário (crescente)
# O(n log n) — algoritmo Merge Sort
# ------------------------------------------------------------
pedidos_ordenados = merge_sort(pedidos_tuplas, index=5, ascending=True) # índice 5 = valor_unitario
print("\n=== Pedidos Ordenados por Valor Unitário ===")
pprint(pedidos_ordenados)


# ------------------------------------------------------------
# 3. ORGANIZAÇÃO DA FILA POR URGÊNCIA (RECURSÃO + DICIONÁRIO DE DEQUES)
# Organiza os pedidos em um dicionário onde cada chave é uma urgência
# e o valor é um deque contendo os respectivos pedidos.
# ------------------------------------------------------------
def organizar_fila(pedidos, resultado_dict=None, prioridades=None):
    """
    Função recursiva que retorna um dicionário com deques separados por urgência.
    """
    if prioridades is None:
        prioridades = ["alta", "media", "baixa"]
    if resultado_dict is None:
        resultado_dict = {p: deque() for p in prioridades}
    if not prioridades:
        return resultado_dict

    categoria_atual = prioridades[0]
    URGENCIA_IDX = 6

    for item in pedidos:
        if item[URGENCIA_IDX] == categoria_atual:
            resultado_dict[categoria_atual].append(item)

    return organizar_fila(pedidos, resultado_dict, prioridades[1:])

# Executa a organização
resultado_final = organizar_fila(pedidos_ordenados)

print("\n=== Dicionário de Pedidos por Urgência ===")
pprint(resultado_final)


# ------------------------------------------------------------
# 3.5 UNIFICAÇÃO DA FILA ÚNICA (USANDO APPEND E APPENDLEFT)
# A partir do dicionário, cria uma fila única estruturada.
# Pega a fila 'media' como base, usa appendleft para a 'alta'
# e append para a 'baixa'.
# ------------------------------------------------------------

# 1. Pega a fila (deque) de média urgência como ponto de partida
fila_organizada = resultado_final["media"]

# 2. Insere a alta urgência no início usando appendleft()
# OBS: Usamos reversed() aqui para não quebrar a ordem do Merge Sort.
# Como o appendleft empurra para trás, iterar de trás pra frente
# garante que o pedido mais barato de alta urgência fique no topo 1.
for item in reversed(resultado_final["alta"]):
    fila_organizada.appendleft(item)

# 3. Insere a baixa urgência no final usando append()
for item in resultado_final["baixa"]:
    fila_organizada.append(item)

print("\n=== Fila Única Organizada (Alta -> Média -> Baixa) ===")
pprint(fila_organizada)


# ------------------------------------------------------------
# 4. GERAÇÃO DE GRÁFICOS
# Visualizações para análise dos dados logísticos
# Big O: O(n) para montagem dos dados em cada gráfico
# ------------------------------------------------------------

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Gráfico 1 — Quantidade de pedidos por urgência (barras)
urgencia_count = df_par['urgencia'].value_counts().reset_index()
urgencia_count.columns = ['urgencia', 'quantidade']

sns.barplot(data=urgencia_count, x='urgencia', y='quantidade', palette='viridis', hue='urgencia', legend=False, ax=axs[0])
axs[0].set_title('Quantidade de Pedidos por Urgência')
axs[0].set_xlabel('Urgência')
axs[0].set_ylabel('Quantidade')
axs[0].tick_params(axis='x', rotation=45)

# Gráfico 2 — Valor total por cidade de destino (barras)
valor_cidade = df_par.groupby('cidade_destino')['valor_unitario'].sum().reset_index()

sns.barplot(data=valor_cidade, x='cidade_destino', y='valor_unitario', palette='pastel', hue='cidade_destino', legend=False, ax=axs[1])
axs[1].set_title('Valor Total por Cidade de Destino')
axs[1].set_xlabel('Cidade')
axs[1].set_ylabel('Valor Total (R$)')
axs[1].tick_params(axis='x', rotation=30)

# Gráfico 3 — Distribuição de quantidade por pedido (pizza)
qtd_cidade = df_par.groupby('cidade_destino')['quantidade'].sum().reset_index()

axs[2].pie(
    qtd_cidade['quantidade'],
    labels=qtd_cidade['cidade_destino'],
    autopct='%1.1f%%',
    colors=sns.color_palette('bright', len(qtd_cidade)),
    startangle=90
)
axs[2].set_title('Distribuição de Quantidade por Cidade')

fig.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'Check_point_1_dados_logistica_RA_final_par.csv'

# Impar

In [ ]:
# ------------------------------------------------------------
# 1. CARREGAMENTO DOS DADOS (ÍMPAR)
# Colunas originais: entrega_id, origem, destino, tipo_carga, peso_kg,
# valor_frete, prazo_dias, prioridade_cliente, situacao_rota, transportadora
# ------------------------------------------------------------
df_impar = pd.read_csv("Check_point_1_dados_logistica_RA_final impar.csv")

# Conversão para tuplas visando imutabilidade das estruturas de dados - O(n)
cargas_tuplas = [tuple(row) for row in df_impar.itertuples(index=False, name=None)]

print("=== DataFrame Ímpar Original ===")
display(df_impar)

# ------------------------------------------------------------
# 2. ORDENAÇÃO MULTICRITÉRIO (Custo, Prazo, Criticidade)
# O(n log n) - Utilização da estabilidade do Merge Sort aplicando
# ordenações sucessivas do critério menos relevante para o mais relevante.
# ------------------------------------------------------------

# Passo 1 (Critério 3 de desempate): Custo (valor_frete - índice 5) -> Maior frete primeiro
ordem_custo = merge_sort(cargas_tuplas, index=5, ascending=False)

# Passo 2 (Critério 2): Prazo (prazo_dias - índice 6) -> Menor prazo primeiro
ordem_prazo = merge_sort(ordem_custo, index=6, ascending=True)

# Passo 3 (Critério 1 - Principal): Criticidade (prioridade_cliente - índice 7)
# 'vip' vem antes de 'normal' na ordem alfabética decrescente (V > N)
cargas_priorizadas = merge_sort(ordem_prazo, index=7, ascending=False)

print("\n=== Top 5 Cargas (Prioridade: VIP > Prazo Curto > Maior Frete) ===")
for c in cargas_priorizadas[:5]:
    print(c)

# ------------------------------------------------------------
# 3. RECURSÃO: CATEGORIZAÇÃO EM DICIONÁRIO DE DEQUES
# ------------------------------------------------------------
def organizar_prioridade_recursivo(pedidos, categorias=None, resultado=None):
    if categorias is None:
        categorias = ["vip", "normal"]
    if resultado is None:
        resultado = {cat: deque() for cat in categorias}
    if not categorias:
        return resultado

    cat_atual = categorias[0]
    PRIO_IDX = 7

    # Inserção nas filas respeitando a ordenação prévia
    for item in pedidos:
        if item[PRIO_IDX] == cat_atual:
            resultado[cat_atual].append(item)

    return organizar_prioridade_recursivo(pedidos, categorias[1:], resultado)


fila_prioridade = organizar_prioridade_recursivo(cargas_priorizadas)

print("\n=== Dicionário de Cargas (Filas Ordenadas) ===")
pprint(fila_prioridade)

# ------------------------------------------------------------
# 4. GERAÇÃO DE GRÁFICOS
# ------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1 - Prazo médio por destino (Barras)
prazo_destino = df_impar.groupby('destino')['prazo_dias'].mean().reset_index()

sns.barplot(data=prazo_destino, x='destino', y='prazo_dias', ax=ax1, palette='magma', hue='destino', legend=False)
ax1.set_title('Prazo Médio de Entrega por Destino')
ax1.set_xlabel('Destino')
ax1.set_ylabel('Prazo Médio (Dias)')

# Gráfico 2 - Relação Peso x Valor do Frete (Dispersão)
sns.scatterplot(data=df_impar, x='peso_kg', y='valor_frete', hue='prioridade_cliente', palette='Set1', s=100, ax=ax2)
ax2.set_title('Relação Peso vs Valor do Frete')
ax2.set_xlabel('Peso (kg)')
ax2.set_ylabel('Valor do Frete (R$)')

plt.tight_layout()
plt.show()